In [1]:
import cv2
import numpy as np
import math
from ultralytics import YOLO

In [2]:
def get_center(box):
    x1, y1, x2, y2 = box
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def smooth(data, window=5):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

In [3]:
VIDEO_PATH = "video1.mp4"
MODEL_PATH = "yolov8s.pt"   
TARGET_CLASS = None

MOVEMENT_THRESHOLD = 2      # pixels
STOP_WINDOW = 5             # frames
SMOOTH_WINDOW = 5

In [4]:
model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

positions = []
times = []
frame_count = 0

In [5]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    detected_center = None

    print('results', results)
    print('frame', frame)

    for r in results:
        for box, cls in zip(r.boxes.xyxy, r.boxes.cls):
            class_name = model.names[int(cls)]
            print(class_name)

            if (class_name != 'bench' and class_name != 'dining table'):
                x1, y1, x2, y2 = map(int, box)
                detected_center = get_center((x1, y1, x2, y2))

                # Draw detection (optional)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.circle(frame, detected_center, 5, (0,0,255), -1)
                break
        if detected_center:
            break

    if detected_center:
        positions.append(detected_center)
        times.append(frame_count / fps)

    frame_count += 1

    # OPTIONAL: display
    cv2.imshow("Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

results [ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potte

In [6]:
if len(positions) < 10:
    print("Not enough detections.")
    exit()

x_vals = [p[0] for p in positions]
y_vals = [p[1] for p in positions]

y_smooth = smooth(y_vals, SMOOTH_WINDOW)
t_smooth = times[:len(y_smooth)]

touch_time = None

for i in range(1, len(y_smooth)):
    prev_y = y_smooth[i-1]
    curr_y = y_smooth[i]

    if curr_y >= prev_y:
        touch_time = t_smooth[i]
        break

stop_time = None

for i in range(STOP_WINDOW, len(positions)):
    movement = 0
    for j in range(i - STOP_WINDOW, i):
        movement += distance(positions[j], positions[j+1])

    if movement < MOVEMENT_THRESHOLD:
        stop_time = times[i]
        break

print("/n===== RESULTS =====")
print("Touch ground at:", touch_time, "seconds")
print("Stop moving at:", stop_time, "seconds")

/n===== RESULTS =====
Touch ground at: 0.5666666666666667 seconds
Stop moving at: 0.7666666666666667 seconds


In [7]:
model.names #13 bench, 60 dining table

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou